# Music Transformer — Training Notebook

**Architecture:** Huang et al. (ICLR 2019) — Relative self-attention  
**Key difference from Step 6:** No GPT-2, no absolute positional encodings.  
This model learns to attend to *relative distances* between tokens, which naturally  
captures musical repetition, phrases, and motifs.

### Two-stage training
1. **Pre-train** on all 666 MIDI files combined (REMI, all traditions) → general music model  
2. **Fine-tune** × 5 traditions × 2 tokenisers (REMI / EC-REMI) → 10 culture-specific models

### Architecture
- 4 decoder layers, d_model=256, 4 heads, d_ff=1024 (~5M params)
- Relative multi-head self-attention with efficient skewing algorithm
- Trained entirely from scratch — custom PyTorch, no HuggingFace Trainer

### Train / Val / Test split
Files are split 80/10/10 per tradition **before any training begins**.  
The test set (10%) is written to disk and never used during training or validation.

**Runtime:** T4 GPU recommended. Pre-training ~45 min, each fine-tune ~10-15 min.


## Cell 1 — Install Dependencies

In [ ]:
!pip install -q miditok==3.0.4 symusic==0.5.6
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Drive and Clone Repo

Replace `REPO_URL` with your GitHub repo URL before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

REPO_URL  = 'https://github.com/AshrafZohdi/Thesis-Best.git'
REPO_DIR  = '/content/Thesis-Best'
DRIVE_OUT = '/content/drive/MyDrive/thesis_music_transformer'
os.makedirs(DRIVE_OUT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('Repo ready:', REPO_DIR)
print('Drive output:', DRIVE_OUT)

## Cell 3 — Imports and Path Setup

In [ ]:
import sys, math, json, random, time, shutil
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = Path(REPO_DIR)

# ── Link your Drive data into the repo so train.py finds it ──────────────────
# Your MIDI files live at My Drive/Thesis-Data/data/processed/{tradition}/midi/
# train.py expects them at /content/Thesis-Best/data/processed/{tradition}/midi/
DRIVE_DATA = Path('/content/drive/MyDrive/Thesis-Data/data')
REPO_DATA  = ROOT / 'data'
if not REPO_DATA.exists():
    REPO_DATA.symlink_to(DRIVE_DATA)
    print(f'Linked data: {DRIVE_DATA} → {REPO_DATA}')
else:
    print(f'Data path ready: {REPO_DATA}')

# Verify MIDI files are accessible
print('\nMIDI file counts per tradition:')
for trad in ['western_classical', 'hindustani', 'carnatic', 'irish_folk', 'turkish_makam']:
    midi_dir = REPO_DATA / 'processed' / trad / 'midi'
    n = len(list(midi_dir.glob('*.mid')) + list(midi_dir.glob('*.midi'))) if midi_dir.exists() else 0
    status = '✓' if n > 0 else '✗ NOT FOUND'
    print(f'  {trad:<25} {n} files  {status}')

# ── sys.path setup ────────────────────────────────────────────────────────────
_ds = str(ROOT / 'datasets')
if _ds in sys.path:
    sys.path.remove(_ds)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from music_transformer.model import MusicTransformer, MusicTransformerConfig
from music_transformer.train import (
    TRADITIONS, CHUNK_SIZE, REMI_VOCAB_SIZE, EC_REMI_VOCAB_SIZE,
    get_split_files, MIDIChunkDataset, get_lr,
    load_remi_tokenizer, load_ec_remi_tokenizer,
)

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SPLIT_DIR = ROOT / 'music_transformer' / 'splits'
CKPT_DIR  = Path(DRIVE_OUT) / 'checkpoints'
GEN_DIR   = Path(DRIVE_OUT) / 'generated'
for d in [CKPT_DIR, GEN_DIR, SPLIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\nDevice      : {DEVICE}')
print(f'Checkpoints : {CKPT_DIR}')
print(f'Generated   : {GEN_DIR}')

## Cell 4 — Data Split Inspection

This creates the 80/10/10 file-level split on first run and saves it to `music_transformer/splits/`.  
The split is deterministic (fixed seed) and stable across runs.

In [ ]:
print('=== Train / Val / Test split (per tradition) ===')
print(f'{"Tradition":<25} {"Train":>6} {"Val":>5} {"Test":>5}')
print('-' * 45)
for trad in TRADITIONS:
    tr = get_split_files(trad, 'train', SPLIT_DIR)
    va = get_split_files(trad, 'val',   SPLIT_DIR)
    te = get_split_files(trad, 'test',  SPLIT_DIR)
    print(f'{trad:<25} {len(tr):>6} {len(va):>5} {len(te):>5}')

# Persist split definitions to Drive so they survive session resets
drive_split = Path(DRIVE_OUT) / 'splits'
drive_split.mkdir(exist_ok=True)
for f in SPLIT_DIR.glob('*.json'):
    shutil.copy(f, drive_split / f.name)
print(f'\nSplit files saved to {drive_split}')

## Cell 5 — Training Helper

In [ ]:
def build_loaders(traditions, tok, tok_type, batch_size=16, tradition_label='western_classical'):
    train_files, val_files = [], []
    for trad in traditions:
        train_files.extend(get_split_files(trad, 'train', SPLIT_DIR))
        val_files.extend(  get_split_files(trad, 'val',   SPLIT_DIR))
    random.shuffle(train_files)
    random.shuffle(val_files)
    print(f'  Files : {len(train_files)} train / {len(val_files)} val')

    train_ds = MIDIChunkDataset(train_files, tok, tok_type, tradition_label,
                                chunk_size=CHUNK_SIZE, stride=CHUNK_SIZE // 2)
    val_ds   = MIDIChunkDataset(val_files,   tok, tok_type, tradition_label,
                                chunk_size=CHUNK_SIZE)
    print(f'  Chunks: {len(train_ds)} train / {len(val_ds)} val')

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2,
                              shuffle=False, num_workers=2)
    return train_loader, val_loader


def run_training(model, train_loader, val_loader,
                 epochs=30, max_lr=3e-4, min_lr=3e-5,
                 warmup_steps=500, grad_clip=1.0, weight_decay=0.01,
                 save_dir=None, log_every=100):
    optimizer  = torch.optim.AdamW(model.parameters(), lr=max_lr,
                                    betas=(0.9, 0.95), weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    warmup      = min(warmup_steps, total_steps // 10)
    use_amp     = (DEVICE.type == 'cuda')
    scaler      = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val, step = float('inf'), 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, t0 = 0.0, time.time()

        for i, (x, y) in enumerate(train_loader, 1):
            x, y = x.to(DEVICE), y.to(DEVICE)
            lr = get_lr(step, warmup, total_steps, max_lr, min_lr)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                _, loss = model(x, targets=y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            step += 1

            if i % log_every == 0:
                print(f'  epoch {epoch:02d} | step {i:04d}/{len(train_loader)} '
                      f'| loss={epoch_loss/i:.4f} | lr={lr:.2e}')

        # Validation
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                with torch.cuda.amp.autocast(enabled=use_amp):
                    _, loss = model(x.to(DEVICE), targets=y.to(DEVICE))
                vl += loss.item()
        vl /= len(val_loader)

        tl  = epoch_loss / len(train_loader)
        ppl = math.exp(min(vl, 10))
        print(f'Epoch {epoch:02d}/{epochs} | train={tl:.4f} val={vl:.4f} ppl={ppl:.1f} '
              f'({time.time()-t0:.0f}s)')
        history.append({'epoch': epoch, 'train_loss': tl, 'val_loss': vl, 'ppl': ppl})

        if save_dir:
            model.save(Path(save_dir) / 'latest')
            if vl < best_val:
                best_val = vl
                model.save(Path(save_dir) / 'best')
                print(f'  ✓ New best val={best_val:.4f}')

    if save_dir:
        with open(Path(save_dir) / 'history.json', 'w') as f:
            json.dump(history, f, indent=2)
    return best_val, history

print('Training helpers ready.')

## Cell 6 — Stage 1: Pre-train on Full MAESTRO Dataset (1,276 files)

Train the Music Transformer on the **complete MAESTRO piano corpus** — not just your 150
cultural files, but all 1,276 recordings (~200 hours of music) already in your Drive.

This is the correct pre-training approach: use a large, diverse music corpus to teach
the model general musical grammar (bars, beats, dynamics, phrase structure), then
fine-tune per culture. This is exactly what Google Magenta did.

The cultural MIDI files are **not used here at all** — they are saved entirely for fine-tuning.

Skip this cell if `checkpoints/pretrain/best/` already exists in Drive.

In [ ]:
PRETRAIN_DIR  = CKPT_DIR / 'pretrain'
MAESTRO_DIR   = Path('/content/drive/MyDrive/Thesis-Data/datasets/western_classical/maestro-v3.0.0')

if (PRETRAIN_DIR / 'best' / 'model.pt').exists():
    print(f'Pre-trained checkpoint found — skipping Cell 6.')
else:
    print('=== Stage 1: Pre-training on full MAESTRO dataset ===')

    # Collect all MIDI files (MAESTRO organises by year subdirectory)
    maestro_files = sorted(
        list(MAESTRO_DIR.rglob('*.midi')) + list(MAESTRO_DIR.rglob('*.mid'))
    )
    print(f'Found {len(maestro_files)} MAESTRO files at {MAESTRO_DIR}')
    if not maestro_files:
        raise FileNotFoundError(
            f'No MIDI files found at {MAESTRO_DIR}\n'
            'Check that your Drive path is correct.'
        )

    # 90 / 10 train / val split (no test set needed for pre-training)
    random.seed(42)
    shuffled = maestro_files.copy()
    random.shuffle(shuffled)
    n_val        = max(1, int(0.10 * len(shuffled)))
    train_files  = shuffled[:-n_val]
    val_files    = shuffled[-n_val:]
    print(f'Split: {len(train_files)} train / {len(val_files)} val')

    # Tokenise and chunk (takes ~2-3 min for 1,276 files)
    print('Tokenising MAESTRO files — please wait...')
    remi_tok = load_remi_tokenizer()
    train_ds = MIDIChunkDataset(train_files, remi_tok, 'remi', 'western_classical',
                                chunk_size=CHUNK_SIZE, stride=CHUNK_SIZE // 2)
    val_ds   = MIDIChunkDataset(val_files,   remi_tok, 'remi', 'western_classical',
                                chunk_size=CHUNK_SIZE)
    print(f'Chunks: {len(train_ds):,} train / {len(val_ds):,} val')

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

    cfg = MusicTransformerConfig(
        vocab_size=REMI_VOCAB_SIZE, d_model=256, n_heads=4,
        n_layers=4, d_ff=1024, max_seq_len=CHUNK_SIZE, dropout=0.1,
    )
    model = MusicTransformer(cfg).to(DEVICE)
    print(f'Model: {model.n_params:,} parameters')

    best_val, history = run_training(
        model, train_loader, val_loader,
        epochs=30, max_lr=3e-4, min_lr=3e-5,
        warmup_steps=1000,   # larger warmup since dataset is much bigger
        save_dir=PRETRAIN_DIR,
    )
    print(f'\nPre-training done. Best val_loss={best_val:.4f}')
    del model
    torch.cuda.empty_cache()

## Cell 7 — Stage 2: Fine-tune (5 Traditions × 2 Tokenisers = 10 Models)

Each model starts from the pre-trained checkpoint.  
For EC-REMI, the vocab is expanded from 284 → 526 tokens before fine-tuning  
(the 242 new cultural token embeddings are randomly initialised).

Models already checkpointed in Drive are skipped automatically.

In [ ]:
from miditok.classes import TokSequence
from symusic import Score as SScore

PRETRAIN_BEST = CKPT_DIR / 'pretrain' / 'best'
assert (PRETRAIN_BEST / 'model.pt').exists(), \
    f'Run Cell 6 first — pre-trained checkpoint not found at {PRETRAIN_BEST}'

results = {}

for tradition in TRADITIONS:
    for tok_type in ['remi', 'ec_remi']:
        run_name = f'{tradition}_{tok_type}'
        save_dir = CKPT_DIR / run_name

        if (save_dir / 'best' / 'model.pt').exists():
            print(f'Skipping {run_name} — already done.')
            h = json.loads((save_dir / 'history.json').read_text()) if (save_dir / 'history.json').exists() else []
            best = min((e['val_loss'] for e in h), default=float('nan'))
            results[run_name] = {'best_val_loss': best, 'ppl': math.exp(min(best, 10))}
            continue

        print(f'\n=== Fine-tuning: {run_name} ===')

        if tok_type == 'remi':
            tok = load_remi_tokenizer()
            vocab_size = REMI_VOCAB_SIZE
        else:
            tok, _ = load_ec_remi_tokenizer(tradition)
            vocab_size = EC_REMI_VOCAB_SIZE

        train_loader, val_loader = build_loaders(
            [tradition], tok, tok_type,
            batch_size=16, tradition_label=tradition
        )

        # Load pre-trained model
        model = MusicTransformer.load(PRETRAIN_BEST, map_location='cpu')
        if tok_type == 'ec_remi':
            model.resize_vocab(EC_REMI_VOCAB_SIZE)
            print(f'  Expanded vocab: {REMI_VOCAB_SIZE} → {EC_REMI_VOCAB_SIZE}')
        model = model.to(DEVICE)

        best_val, history = run_training(
            model, train_loader, val_loader,
            epochs=20, max_lr=1e-4, min_lr=1e-5,
            warmup_steps=100, save_dir=save_dir,
        )
        results[run_name] = {'best_val_loss': best_val, 'ppl': math.exp(min(best_val, 10))}

        del model
        torch.cuda.empty_cache()

# Print summary
print('\n=== Fine-tuning Summary ===')
print(f'{"Model":<35} {"Val Loss":>10} {"PPL":>8}')
print('-' * 57)
for run, r in sorted(results.items()):
    print(f'{run:<35} {r["best_val_loss"]:>10.4f} {r["ppl"]:>8.1f}')

with open(CKPT_DIR / 'finetune_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

## Cell 8 — Generate MIDI Samples

For each of the 10 models, generate 5 MIDI files.  
Seed prompts come from the **test split** — files the model has never seen.

In [ ]:
from miditok.classes import TokSequence
from symusic import Score as SScore

N_SAMPLES   = 5
GEN_TOKENS  = 512
SEED_LEN    = 64
TEMPERATURE = 0.92
TOP_P       = 0.92

remi_dec = load_remi_tokenizer()   # always used for decoding


def get_seed_ids(tradition, tok, tok_type, idx=0):
    try:
        files = get_split_files(tradition, 'test', SPLIT_DIR)
        if not files:
            return None
        fpath = files[idx % len(files)]
        if tok_type == 'remi':
            seqs = tok.encode(SScore(str(fpath)))
            if seqs and len(seqs[0].ids) >= SEED_LEN:
                return seqs[0].ids[:SEED_LEN]
        else:
            toks = tok.tokenize(fpath, tradition=tradition)
            if toks:
                ids = tok.encode(toks)
                if len(ids) >= SEED_LEN:
                    return ids[:SEED_LEN]
    except Exception as e:
        print(f'    seed error: {e}')
    return None


def ids_to_midi(ids, out_path, tok_type):
    try:
        valid = [i for i in ids if 0 <= i < REMI_VOCAB_SIZE]
        if len(valid) < 5:
            return False
        score = remi_dec.decode([TokSequence(ids=valid)])
        if not score.tracks or not any(len(t.notes) > 0 for t in score.tracks):
            return False
        score.dump_midi(str(out_path))
        return True
    except Exception as e:
        print(f'    decode error: {e}')
        return False


gen_summary = {}

for tradition in TRADITIONS:
    for tok_type in ['remi', 'ec_remi']:
        run_name  = f'{tradition}_{tok_type}'
        ckpt_path = CKPT_DIR / run_name / 'best'

        if not (ckpt_path / 'model.pt').exists():
            print(f'Skipping {run_name} — no checkpoint')
            continue

        print(f'\nGenerating: {run_name}')
        model = MusicTransformer.load(ckpt_path, map_location='cpu').to(DEVICE)

        tok = load_remi_tokenizer() if tok_type == 'remi' else load_ec_remi_tokenizer(tradition)[0]

        out_dir = GEN_DIR / run_name
        out_dir.mkdir(parents=True, exist_ok=True)

        saved = 0
        for i in range(N_SAMPLES):
            seed = get_seed_ids(tradition, tok, tok_type, idx=i)
            if seed is None:
                print(f'  [{i}] no seed')
                continue

            torch.manual_seed(42 + i)
            prompt = torch.tensor([seed], dtype=torch.long, device=DEVICE)
            output = model.generate(prompt, GEN_TOKENS, TEMPERATURE, TOP_P)
            ids    = output[0].cpu().tolist()

            midi_path = out_dir / f'sample_{i:02d}.mid'
            ok = ids_to_midi(ids, midi_path, tok_type)
            print(f'  [{i}] {len(ids)} tokens → {"saved" if ok else "decode failed"}')
            if ok:
                saved += 1

        gen_summary[run_name] = {'saved': saved, 'total': N_SAMPLES}
        del model
        torch.cuda.empty_cache()

print('\n=== Generation Summary ===')
for run, s in gen_summary.items():
    print(f'  {run:<35} {s["saved"]}/{s["total"]} MIDI files')

## Cell 9 — Save Split Definitions and Print Final Report

In [ ]:
# Persist split definitions
drive_split = Path(DRIVE_OUT) / 'splits'
drive_split.mkdir(exist_ok=True)
for f in SPLIT_DIR.glob('*.json'):
    shutil.copy(f, drive_split / f.name)

# Final report
summary_path = CKPT_DIR / 'finetune_summary.json'
if summary_path.exists():
    results = json.loads(summary_path.read_text())
    print('=== Music Transformer — Final Results ===')
    print(f'{"Model":<35} {"Val Loss":>10} {"PPL":>8} {"Ckpt":>6}')
    print('-' * 63)
    for tradition in TRADITIONS:
        for tok_type in ['remi', 'ec_remi']:
            run = f'{tradition}_{tok_type}'
            r   = results.get(run, {})
            ok  = (CKPT_DIR / run / 'best' / 'model.pt').exists()
            vl  = r.get('best_val_loss', float('nan'))
            ppl = r.get('ppl', float('nan'))
            print(f'{run:<35} {vl:>10.4f} {ppl:>8.1f} {"✓" if ok else "✗":>6}')

print('\n=== Test Split (held-out, never trained on) ===')
for trad in TRADITIONS:
    files = get_split_files(trad, 'test', SPLIT_DIR)
    print(f'  {trad:<25} {len(files)} files')

print(f'\nAll outputs at: {DRIVE_OUT}')